In [21]:
import fitness.data_generating_process as dgp
import fitness.soga_fitness_SCM as soga_fitness_SCM
import fitness.interventions as interventions
import pandas as pd
import numpy as np

In [ ]:
program = 'chain'
vars, scm = dgp.get_vars(program)
real_program = dgp.get_real_program(program)

real_dataset = dgp.generate_interventional_dataset(scm, vars, 10000)

# from the real dataset, print mean and std for each variable

df = pd.DataFrame(real_dataset, columns=vars)
print(df.describe())
np.savetxt(f'fitness/datasets/{program}.csv', real_dataset, delimiter=',')

                  F             C             W             T             P
count  10000.000000  10000.000000  10000.000000  10000.000000  10000.000000
mean      10.003261      4.992853      4.986771     14.988299    237.943949
std        1.997170      1.151099      2.897316      3.645276    110.999161
min        1.868203      0.530035      0.001298      1.674789     -1.907601
25%        8.673929      4.206738      2.466530     12.293358    150.565596
50%       10.025082      5.028965      4.997461     14.989624    224.363580
75%       11.369621      5.776824      7.490857     17.703726    315.251070
max       17.043953      9.217501      9.999163     26.925327    715.441801


In [23]:
baseline_program = dgp.get_baseline_program(program)

In [24]:
#LIKELIHOOD WITHOUT INTERVENTIONS:
likelihood = soga_fitness_SCM.compute_likelihood(real_program, vars, real_dataset)
print(f'Likelihood without interventions of real program: {likelihood}')

likelihood_base = soga_fitness_SCM.compute_likelihood(baseline_program, vars, real_dataset)
print(f'Likelihood without interventions of baseline program: {likelihood_base}')

Likelihood without interventions of real program: -11.198966451822992
Likelihood without interventions of baseline program: -14.99391583977543


In [ ]:

interventions_list =  [("F", 5.0), ("F", 15.0), ("C", 2.0), ("C", 7.0),  ("W", 2.0), ("W", 7.0), ("T", 25.0), ("T", 7.0), ("P", 200.0), ("P", 700.0)]


for var, value in interventions_list:
    data_intervened = dgp.generate_interventional_dataset(scm, vars, 100, intervention={var: value})
    # save dataset like ../datasets/{program}_intervention_{var}_{value}.csv', delimiter=',' creating a new file if it does not exist
    np.savetxt(f'fitness/datasets/{program}_intervention_{var}_{value}.csv', data_intervened, delimiter=',')

    program_intervened = interventions.apply_intervention_to_program(real_program, var, value)
    #print(program_intervened)
    baseline_program_intervened = interventions.apply_intervention_to_program(baseline_program, var, value)
    #print(baseline_program_intervened)
    interventional_likelihood = soga_fitness_SCM.compute_likelihood(program_intervened, vars, data_intervened)
    interventional_likelihood_base = soga_fitness_SCM.compute_likelihood(baseline_program_intervened, vars, data_intervened)
    #print(f'Interventional likelihood of real program after intervening {var}={value}: {interventional_likelihood}')
    #print(f'Interventional likelihood of baseline program after intervening {var}={value}: {interventional_likelihood_base}')
    likelihood += interventional_likelihood
    likelihood_base += interventional_likelihood_base


print(f'Total likelihood of real program: {likelihood}')
print(f'Total likelihood of baseline program: {likelihood_base}')

Total likelihood of real program: -99.38371897494508
Total likelihood of baseline program: -148.50287031892336
